# Práctica SQL 02 — Caso 02: Academia Online (preguntas)

10 preguntas básicas y 20 intermedias.

**Plataforma:** Databricks SQL · Delta Lake · Unity Catalog

## Descripción

Una plataforma latinoamericana ofrece cursos de tecnología, negocios, marketing y diseño. La empresa desea analizar estudiantes, instructores, inscripciones, pagos, sesiones de estudio y reseñas para mejorar el aprendizaje y los ingresos.

### Reglas de negocio para los ejercicios

- Considera como ingreso únicamente los pagos con `estado_pago = 'aprobado'`.
- Una inscripción completada tiene `estado = 'completado'`.
- La tasa de finalización se calcula como inscripciones completadas entre inscripciones totales.

## Tablas

| Tabla | Columnas y tipos | Descripción |
|---|---|---|
| `estudiantes` | `estudiante_id STRING`; `nombre_estudiante STRING`; `pais STRING`; `ciudad STRING`; `plan STRING`; `fecha_registro DATE`; `edad INT` | País, plan y fecha de registro. |
| `instructores` | `instructor_id STRING`; `nombre_instructor STRING`; `pais STRING`; `especialidad STRING`; `fecha_ingreso DATE` | Especialidad y fecha de ingreso. |
| `cursos` | `curso_id STRING`; `instructor_id STRING`; `titulo_curso STRING`; `categoria STRING`; `nivel STRING`; `precio DECIMAL(10,2)`; `duracion_horas DECIMAL(8,2)`; `fecha_publicacion DATE`; `activo BOOLEAN` | Categoría, nivel, precio, duración e instructor. |
| `inscripciones` | `inscripcion_id STRING`; `estudiante_id STRING`; `curso_id STRING`; `fecha_inscripcion DATE`; `estado STRING`; `porcentaje_avance INT`; `nota_final INT` *(admite NULL)* | Relación estudiante-curso, avance y nota. |
| `pagos` | `pago_id STRING`; `inscripcion_id STRING`; `fecha_pago DATE`; `monto DECIMAL(10,2)`; `metodo_pago STRING`; `estado_pago STRING` | Monto, método y estado del pago. |
| `sesiones` | `sesion_id STRING`; `inscripcion_id STRING`; `fecha_sesion DATE`; `minutos_vistos INT`; `dispositivo STRING` | Actividad de estudio por inscripción. |
| `resenas` | `resena_id STRING`; `inscripcion_id STRING`; `fecha_resena DATE`; `calificacion INT`; `comentario STRING` | Calificación y comentario de cursos completados. |

## Relaciones

- `instructores.instructor_id` → `cursos.instructor_id` (1 a N)
- `estudiantes.estudiante_id` → `inscripciones.estudiante_id` (1 a N)
- `cursos.curso_id` → `inscripciones.curso_id` (1 a N)
- `inscripciones.inscripcion_id` → `pagos.inscripcion_id` (1 a N)
- `inscripciones.inscripcion_id` → `sesiones.inscripcion_id` (1 a N)
- `inscripciones.inscripcion_id` → `resenas.inscripcion_id` (1 a N)

## Consideraciones

- Los archivos TXT usan formato CSV con encabezado y codificación UTF-8.
- Las fechas usan el formato `YYYY-MM-DD`.
- `nota_final` solo se completa cuando la inscripción está terminada (en otro caso es `NULL`).
- Los cursos gratuitos tienen pagos aprobados por monto `0.00` para conservar la trazabilidad.

---
## PARTE 0 — Preparación del entorno (DDL y carga)

> Si ya creaste y cargaste las tablas de este caso, salta directamente a la **PARTE 1**.

### 0.1 Esquemas y volumen

| Objeto | Nombre | Para qué sirve |
|---|---|---|
| Esquema | `workspace.sql_practice_raw` | Contiene el volumen con los archivos originales. |
| Volumen | `workspace.sql_practice_raw.datasets` | Almacena los `.txt` de ambos casos, cada uno en su carpeta. |
| Esquema | `workspace.sql_practice_academia` | Contiene las 7 tablas de este caso. |

En Databricks Free Edition el catálogo por defecto es `workspace`. Si el tuyo se llama `main` u otro nombre, reemplaza `workspace` en todo el notebook, incluidas las rutas `/Volumes/workspace/...`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_raw;
CREATE VOLUME IF NOT EXISTS workspace.sql_practice_raw.datasets;

CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_academia;

### 0.2 Subir los archivos al volumen

1. En **Catalog Explorer** abre el volumen `workspace` → `sql_practice_raw` → **Volumes** → `datasets`.
2. Crea la carpeta **`caso_02_academia_online`** (si aún no existe).
3. Dentro de esa carpeta, sube los 7 archivos de `datos_txt/`: `estudiantes.txt`, `instructores.txt`, `cursos.txt`, `inscripciones.txt`, `pagos.txt`, `sesiones.txt`, `resenas.txt`.
4. Ejecuta la siguiente celda: deben aparecer los 7 archivos.

In [0]:
%sql
LIST '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/';

### 0.3 Eliminar tablas previas

Permite volver a ejecutar la preparación desde cero. Las tablas se eliminan en **orden inverso a sus dependencias**: primero las hijas y luego las padres.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_academia;

DROP TABLE IF EXISTS resenas;
DROP TABLE IF EXISTS sesiones;
DROP TABLE IF EXISTS pagos;
DROP TABLE IF EXISTS inscripciones;
DROP TABLE IF EXISTS cursos;
DROP TABLE IF EXISTS instructores;
DROP TABLE IF EXISTS estudiantes;

### 0.4 Crear las tablas (DDL)

- Todas las columnas son `NOT NULL`, salvo las indicadas como *admite NULL*. Delta Lake **sí** valida esta restricción al insertar.
- `PRIMARY KEY` y `FOREIGN KEY` en Unity Catalog son restricciones **informativas**: documentan el modelo, pero **no se validan** al insertar. Por eso en el paso 0.6 verificamos la integridad con consultas.
- Las tablas se crean de **padres a hijas**: una clave foránea solo puede apuntar a una tabla que ya tenga su clave primaria.

#### `estudiantes`

In [0]:
%sql
--CREATE OR REPLACE -> EQUIVALENTE A DROPEAR Y VOLVER A CREAR 
CREATE TABLE estudiantes (
  estudiante_id      STRING  NOT NULL COMMENT 'Identificador del estudiante',
  nombre_estudiante  STRING  NOT NULL COMMENT 'Nombre completo',
  pais               STRING  NOT NULL COMMENT 'País de residencia',
  ciudad             STRING  NOT NULL COMMENT 'Ciudad de residencia',
  plan               STRING  NOT NULL COMMENT 'Plan de suscripción',
  fecha_registro     DATE    NOT NULL COMMENT 'Fecha de registro',
  edad               INT     NOT NULL COMMENT 'Edad en años',
  CONSTRAINT pk_estudiantes PRIMARY KEY (estudiante_id)
)
COMMENT 'Estudiantes de la plataforma.';

#### `instructores`

In [0]:
%sql
CREATE TABLE instructores (
  instructor_id      STRING  NOT NULL COMMENT 'Identificador del instructor',
  nombre_instructor  STRING  NOT NULL COMMENT 'Nombre completo',
  pais               STRING  NOT NULL COMMENT 'País de residencia',
  especialidad       STRING  NOT NULL COMMENT 'Especialidad del instructor',
  fecha_ingreso      DATE    NOT NULL COMMENT 'Fecha de ingreso a la plataforma',
  CONSTRAINT pk_instructores PRIMARY KEY (instructor_id)
)
COMMENT 'Instructores y su especialidad.';

#### `cursos`

In [0]:
%sql
CREATE TABLE cursos (
  curso_id           STRING         NOT NULL COMMENT 'Identificador del curso',
  instructor_id      STRING         NOT NULL COMMENT 'Instructor a cargo',
  titulo_curso       STRING         NOT NULL COMMENT 'Título del curso',
  categoria          STRING         NOT NULL COMMENT 'Categoría del curso',
  nivel              STRING         NOT NULL COMMENT 'Nivel: Básico, Intermedio o Avanzado',
  precio             DECIMAL(10,2)  NOT NULL COMMENT 'Precio en dólares (0 = gratuito)',
  duracion_horas     DECIMAL(8,2)   NOT NULL COMMENT 'Duración del contenido en horas',
  fecha_publicacion  DATE           NOT NULL COMMENT 'Fecha de publicación',
  activo             BOOLEAN        NOT NULL COMMENT 'Indica si el curso está activo',
  CONSTRAINT pk_cursos PRIMARY KEY (curso_id),
  CONSTRAINT fk_cursos_instructores FOREIGN KEY (instructor_id) REFERENCES instructores (instructor_id)
)
COMMENT 'Catálogo de cursos.';

#### `inscripciones`

In [0]:
%sql
CREATE TABLE inscripciones (
  inscripcion_id     STRING  NOT NULL COMMENT 'Identificador de la inscripción',
  estudiante_id      STRING  NOT NULL COMMENT 'Estudiante inscrito',
  curso_id           STRING  NOT NULL COMMENT 'Curso inscrito',
  fecha_inscripcion  DATE    NOT NULL COMMENT 'Fecha de inscripción',
  estado             STRING  NOT NULL COMMENT 'Estado de la inscripción',
  porcentaje_avance  INT     NOT NULL COMMENT 'Avance del curso (0 a 100)',
  nota_final         INT              COMMENT 'Nota final (NULL si no ha terminado)',
  CONSTRAINT pk_inscripciones PRIMARY KEY (inscripcion_id),
  CONSTRAINT fk_inscripciones_estudiantes FOREIGN KEY (estudiante_id) REFERENCES estudiantes (estudiante_id),
  CONSTRAINT fk_inscripciones_cursos FOREIGN KEY (curso_id) REFERENCES cursos (curso_id)
)
COMMENT 'Relación estudiante-curso.';

#### `pagos`

In [0]:
%sql
CREATE TABLE pagos (
  pago_id         STRING         NOT NULL COMMENT 'Identificador del pago',
  inscripcion_id  STRING         NOT NULL COMMENT 'Inscripción pagada',
  fecha_pago      DATE           NOT NULL COMMENT 'Fecha del pago',
  monto           DECIMAL(10,2)  NOT NULL COMMENT 'Monto pagado',
  metodo_pago     STRING         NOT NULL COMMENT 'Método de pago',
  estado_pago     STRING         NOT NULL COMMENT 'Estado del pago',
  CONSTRAINT pk_pagos PRIMARY KEY (pago_id),
  CONSTRAINT fk_pagos_inscripciones FOREIGN KEY (inscripcion_id) REFERENCES inscripciones (inscripcion_id)
)
COMMENT 'Pagos de cada inscripción.';

#### `sesiones`

In [0]:
%sql
CREATE TABLE sesiones (
  sesion_id       STRING  NOT NULL COMMENT 'Identificador de la sesión',
  inscripcion_id  STRING  NOT NULL COMMENT 'Inscripción a la que pertenece',
  fecha_sesion    DATE    NOT NULL COMMENT 'Fecha de la sesión',
  minutos_vistos  INT     NOT NULL COMMENT 'Minutos de contenido vistos',
  dispositivo     STRING  NOT NULL COMMENT 'Dispositivo utilizado',
  CONSTRAINT pk_sesiones PRIMARY KEY (sesion_id),
  CONSTRAINT fk_sesiones_inscripciones FOREIGN KEY (inscripcion_id) REFERENCES inscripciones (inscripcion_id)
)
COMMENT 'Sesiones de estudio por inscripción.';

#### `resenas`

In [0]:
%sql
CREATE TABLE resenas (
  resena_id       STRING  NOT NULL COMMENT 'Identificador de la reseña',
  inscripcion_id  STRING  NOT NULL COMMENT 'Inscripción reseñada',
  fecha_resena    DATE    NOT NULL COMMENT 'Fecha de la reseña',
  calificacion    INT     NOT NULL COMMENT 'Calificación de 1 a 5',
  comentario      STRING  NOT NULL COMMENT 'Comentario del estudiante',
  CONSTRAINT pk_resenas PRIMARY KEY (resena_id),
  CONSTRAINT fk_resenas_inscripciones FOREIGN KEY (inscripcion_id) REFERENCES inscripciones (inscripcion_id)
)
COMMENT 'Reseñas de cursos completados.';

### 0.5 Cargar los datos

Usamos `read_files` para leer cada `.txt` del volumen:

- `format => 'csv'` y `header => true`: texto delimitado por comas con encabezado.
- `schema => '...'`: define el tipo de cada columna en lugar de dejar que Databricks lo infiera. Los campos vacíos se cargan como `NULL`.
- `mode => 'FAILFAST'`: si alguna fila no se puede convertir al tipo indicado, la carga falla en lugar de dejar datos incorrectos.

#### `estudiantes`

In [0]:
%sql
INSERT INTO estudiantes
SELECT
  estudiante_id,
  nombre_estudiante,
  pais,
  ciudad,
  plan,
  fecha_registro,
  edad
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/estudiantes.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'estudiante_id STRING, nombre_estudiante STRING, pais STRING, ciudad STRING, plan STRING, fecha_registro DATE, edad INT'
);

SELECT * FROM estudiantes LIMIT 5;

#### `instructores`

In [0]:
%sql
INSERT INTO instructores
SELECT
  instructor_id,
  nombre_instructor,
  pais,
  especialidad,
  fecha_ingreso
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/instructores.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'instructor_id STRING, nombre_instructor STRING, pais STRING, especialidad STRING, fecha_ingreso DATE'
);

SELECT * FROM instructores LIMIT 5;

#### `cursos`

In [0]:
%sql
INSERT INTO cursos
SELECT
  curso_id,
  instructor_id,
  titulo_curso,
  categoria,
  nivel,
  precio,
  duracion_horas,
  fecha_publicacion,
  activo
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/cursos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'curso_id STRING, instructor_id STRING, titulo_curso STRING, categoria STRING, nivel STRING, precio DECIMAL(10,2), duracion_horas DECIMAL(8,2), fecha_publicacion DATE, activo BOOLEAN'
);

SELECT * FROM cursos LIMIT 5;

#### `inscripciones`

In [0]:
%sql
INSERT INTO inscripciones
SELECT
  inscripcion_id,
  estudiante_id,
  curso_id,
  fecha_inscripcion,
  estado,
  porcentaje_avance,
  nota_final
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/inscripciones.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'inscripcion_id STRING, estudiante_id STRING, curso_id STRING, fecha_inscripcion DATE, estado STRING, porcentaje_avance INT, nota_final INT'
);

SELECT * FROM inscripciones LIMIT 5;

#### `pagos`

In [0]:
%sql
INSERT INTO pagos
SELECT
  pago_id,
  inscripcion_id,
  fecha_pago,
  monto,
  metodo_pago,
  estado_pago
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/pagos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'pago_id STRING, inscripcion_id STRING, fecha_pago DATE, monto DECIMAL(10,2), metodo_pago STRING, estado_pago STRING'
);

SELECT * FROM pagos LIMIT 5;

#### `sesiones`

In [0]:
%sql
INSERT INTO sesiones
SELECT
  sesion_id,
  inscripcion_id,
  fecha_sesion,
  minutos_vistos,
  dispositivo
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/sesiones.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'sesion_id STRING, inscripcion_id STRING, fecha_sesion DATE, minutos_vistos INT, dispositivo STRING'
);

SELECT * FROM sesiones LIMIT 5;

#### `resenas`

In [0]:
%sql
INSERT INTO resenas
SELECT
  resena_id,
  inscripcion_id,
  fecha_resena,
  calificacion,
  comentario
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/resenas.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'resena_id STRING, inscripcion_id STRING, fecha_resena DATE, calificacion INT, comentario STRING'
);

SELECT * FROM resenas LIMIT 5;

### 0.6 Validación

Todas las tablas deben mostrar `estado = OK`.

In [0]:
%sql
WITH conteos AS (
  SELECT 'estudiantes' AS tabla, COUNT(*) AS filas, 180 AS filas_esperadas FROM estudiantes
  UNION ALL
  SELECT 'instructores' AS tabla, COUNT(*) AS filas, 20 AS filas_esperadas FROM instructores
  UNION ALL
  SELECT 'cursos' AS tabla, COUNT(*) AS filas, 45 AS filas_esperadas FROM cursos
  UNION ALL
  SELECT 'inscripciones' AS tabla, COUNT(*) AS filas, 520 AS filas_esperadas FROM inscripciones
  UNION ALL
  SELECT 'pagos' AS tabla, COUNT(*) AS filas, 520 AS filas_esperadas FROM pagos
  UNION ALL
  SELECT 'sesiones' AS tabla, COUNT(*) AS filas, 4329 AS filas_esperadas FROM sesiones
  UNION ALL
  SELECT 'resenas' AS tabla, COUNT(*) AS filas, 135 AS filas_esperadas FROM resenas
)
SELECT
  tabla,
  filas,
  filas_esperadas,
  IF(filas = filas_esperadas, 'OK', 'REVISAR') AS estado
FROM conteos;

Integridad referencial: como las claves foráneas son informativas, comprobamos que no existan filas hijas que apunten a un padre inexistente. Todas las relaciones deben mostrar `huerfanos = 0`.

In [0]:
%sql
SELECT 'cursos.instructor_id -> instructores' AS relacion, COUNT(*) AS huerfanos
FROM cursos AS h
LEFT JOIN instructores AS p ON h.instructor_id = p.instructor_id
WHERE p.instructor_id IS NULL
UNION ALL
SELECT 'inscripciones.estudiante_id -> estudiantes' AS relacion, COUNT(*) AS huerfanos
FROM inscripciones AS h
LEFT JOIN estudiantes AS p ON h.estudiante_id = p.estudiante_id
WHERE p.estudiante_id IS NULL
UNION ALL
SELECT 'inscripciones.curso_id -> cursos' AS relacion, COUNT(*) AS huerfanos
FROM inscripciones AS h
LEFT JOIN cursos AS p ON h.curso_id = p.curso_id
WHERE p.curso_id IS NULL
UNION ALL
SELECT 'pagos.inscripcion_id -> inscripciones' AS relacion, COUNT(*) AS huerfanos
FROM pagos AS h
LEFT JOIN inscripciones AS p ON h.inscripcion_id = p.inscripcion_id
WHERE p.inscripcion_id IS NULL
UNION ALL
SELECT 'sesiones.inscripcion_id -> inscripciones' AS relacion, COUNT(*) AS huerfanos
FROM sesiones AS h
LEFT JOIN inscripciones AS p ON h.inscripcion_id = p.inscripcion_id
WHERE p.inscripcion_id IS NULL
UNION ALL
SELECT 'resenas.inscripcion_id -> inscripciones' AS relacion, COUNT(*) AS huerfanos
FROM resenas AS h
LEFT JOIN inscripciones AS p ON h.inscripcion_id = p.inscripcion_id
WHERE p.inscripcion_id IS NULL;

---
## Configuración para resolver las preguntas

Ejecuta esta celda cada vez que abras el notebook: las consultas usan nombres de tablas sin catálogo ni esquema.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_academia;

SHOW TABLES;

---
## PARTE 1 — Preguntas básicas (1-10)

### 1. Todos los estudiantes

Muestra todos los registros de `estudiantes`.

In [0]:
%sql
SELECT *
FROM estudiantes;

### 2. Cursos activos

Lista el ID, título, categoría y precio de los cursos activos.

In [0]:
%sql
SELECT curso_id, titulo_curso, categoria, precio
FROM cursos
WHERE activo;

### 3. Categorías únicas

Obtén las categorías únicas de cursos, ordenadas alfabéticamente.

In [0]:
%sql
SELECT DISTINCT categoria
FROM cursos
ORDER BY categoria;

### 4. Cursos básicos ordenados por precio

Muestra los cursos de nivel `Básico`, ordenados por precio de menor a mayor.

In [0]:
%sql
SELECT *
FROM cursos
WHERE nivel = "Básico"
ORDER BY precio;

### 5. Cantidad de instructores

Cuenta cuántos instructores existen.

In [0]:
%sql
SELECT COUNT(1) AS total_instructores
FROM instructores;

### 6. Duración promedio de cursos

Calcula la duración promedio de los cursos.

In [0]:
%sql
SELECT AVG(duracion_horas) AS duracion_promedio_horas
FROM cursos;

### 7. Inscripciones completadas

Muestra todas las inscripciones completadas.

In [0]:
%sql
SELECT *
FROM inscripciones
WHERE estado = 'completado';

### 8. Pagos aprobados mayores a 50

Lista los pagos aprobados cuyo monto sea mayor a 50.

In [0]:
%sql
SELECT *
FROM pagos
WHERE estado_pago = 'aprobado' AND monto > 50
ORDER BY monto;

### 9. Sesiones desde móvil

Muestra las sesiones realizadas desde un dispositivo `Móvil`.

In [0]:
%sql
SELECT *
FROM sesiones
WHERE dispositivo = 'Móvil';

### 10. Cantidad de reseñas por calificación

Cuenta cuántas reseñas existen por cada calificación presente en los datos.

In [0]:
%sql

SELECT
    calificacion,
    COUNT(1) AS cantidad_resenas
FROM resenas
GROUP BY calificacion
ORDER BY calificacion;

---
## PARTE 2 — Preguntas intermedias (11-30)

### 11. Cursos con cantidad de inscripciones, incluyendo cursos sin inscripciones

Muestra cada curso con su cantidad total de inscripciones, incluyendo cursos sin inscripciones.

In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    COUNT(1) AS cantidad_inscripciones
FROM cursos AS c
LEFT JOIN inscripciones AS i
ON c.curso_id = i.curso_id
GROUP BY c.curso_id, c.titulo_curso
ORDER BY cantidad_inscripciones DESC;

### 12. Inscripciones y tasa de finalización por categoría

Calcula por categoría la cantidad de inscripciones, completadas y la tasa de finalización.
- La tasa de finalización se calcula como inscripciones completadas entre inscripciones totales.

In [0]:
%sql
WITH info_inscripciones AS (
    SELECT
        c.categoria,
        COUNT(1) AS inscripciones_totales,
        SUM(
            CASE
                WHEN i.estado = 'completado' THEN 1
                ELSE 0
            END
        ) AS inscripciones_completadas
        -- inscripciones_completadas / inscripciones_totales
    FROM cursos AS c
    LEFT JOIN inscripciones AS i
    USING (curso_id)
    GROUP BY c.categoria
)
SELECT 
    *,
    inscripciones_completadas / inscripciones_totales AS tasa_finalizacion
FROM info_inscripciones
ORDER BY tasa_finalizacion;


### 13. Ingresos aprobados por curso

Calcula los ingresos aprobados por curso.

In [0]:
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(
        CASE
            WHEN p.estado_pago = "aprobado" THEN p.monto
            ELSE 0
        END
    ) ingreso_total
FROM cursos AS c
LEFT JOIN inscripciones AS i
USING (curso_id)
LEFT JOIN pagos AS p
USING (inscripcion_id)
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

In [0]:
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(
        CASE
            WHEN p.estado_pago = "aprobado" THEN p.monto
            ELSE 0
        END
    ) ingreso_total
FROM cursos AS c
LEFT JOIN inscripciones AS i
ON c.curso_id = i.curso_id
LEFT JOIN pagos AS p
ON i.inscripcion_id = p.inscripcion_id
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

In [0]:
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id AND p.estado_pago = "aprobado"
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

In [0]:
WITH pagos_delimitado AS (
    SELECT *
    FROM pagos
    WHERE estado_pago = "aprobado"
)
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos_delimitado AS p
    ON i.inscripcion_id = p.inscripcion_id
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

In [0]:
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id
WHERE p.estado_pago = "aprobado"
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

### 14. Top 10 cursos por ingresos aprobados

Identifica los 10 cursos con mayores ingresos aprobados.

In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id
WHERE p.estado_pago = "aprobado"
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC
LIMIT 10;

### 15. Métricas por instructor

Muestra por instructor: cantidad de cursos, inscripciones totales e ingresos aprobados.

In [0]:
%sql
SELECT
    nombre_instructor,
    COUNT(DISTINCT c.curso_id) AS cantidad_cursos,
    COUNT(DISTINCT i.inscripcion_id) AS inscripciones_totales,
    SUM(p.monto) AS ingresos_totales
FROM instructores AS ins
INNER JOIN cursos AS c
ON ins.instructor_id = c.instructor_id
INNER JOIN inscripciones AS i
ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
ON i.inscripcion_id = P.inscripcion_id AND p.estado_pago = "aprobado"
GROUP BY ins.nombre_instructor;


### 16. Estudiantes sin inscripciones

Lista los estudiantes que nunca se inscribieron en un curso.

In [0]:
%sql
SELECT 
    e.estudiante_id,
    e.nombre_estudiante,
    e.pais
FROM estudiantes AS e
LEFT ANTI JOIN inscripciones AS i
ON e.estudiante_id = i.estudiante_id

### 17. Cursos sin inscripciones

Lista los cursos que nunca recibieron una inscripción.

In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    c.categoria
FROM cursos AS c
LEFT ANTI JOIN inscripciones AS i
ON c.curso_id = i.curso_id

### 18. Nota promedio por nivel para cursos completados

Calcula la nota final promedio de los cursos completados, agrupada por nivel.

In [0]:
%sql
SELECT
    c.nivel,
    AVG(i.nota_final) AS nota_promedio
FROM inscripciones AS i
INNER JOIN cursos AS c
ON i.curso_id = c.curso_id AND i.estado = "completado"
GROUP BY c.nivel

### 19. Minutos vistos por estudiante

Calcula el total de minutos vistos por estudiante y ordénalos de mayor a menor.

In [0]:
%sql
SELECT
    e.nombre_estudiante,
    SUM(COALESCE(s.minutos_vistos, 0)) AS total_minutos
FROM estudiantes AS e
LEFT JOIN inscripciones AS i
ON e.estudiante_id = i.estudiante_id
LEFT JOIN sesiones AS s
ON i.inscripcion_id = s.inscripcion_id
GROUP BY e.nombre_estudiante
ORDER BY total_minutos DESC

### 20. Calificación promedio y cantidad de reseñas por curso

Calcula por curso la calificación promedio y cantidad de reseñas, incluyendo cursos sin reseñas.

In [0]:
%sql
SELECT
    c.titulo_curso,
    AVG(i.nota_final) AS calificacion_promedio,
    COUNT(r.resena_id) AS cantidad_resenas
FROM cursos AS c
LEFT JOIN inscripciones AS i
ON i.curso_id = c.curso_id
LEFT JOIN resenas AS r
ON i.inscripcion_id = r.inscripcion_id
GROUP BY c.titulo_curso;

### 21. Nuevos estudiantes por mes y acumulado

Calcula los nuevos estudiantes registrados por mes y el acumulado histórico.

In [0]:
%sql
WITH historico AS (
    SELECT
        DATE_TRUNC("MONTH",fecha_registro) AS mes,
        COUNT(1) AS numero_estudiantes
    FROM estudiantes
    GROUP BY DATE_TRUNC("MONTH",fecha_registro)
)
SELECT
    mes,
    numero_estudiantes,
    SUM(numero_estudiantes) OVER(
        ORDER BY mes
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW 
    ) AS acumulado
FROM historico
ORDER BY mes;

### 22. Ingresos mensuales y crecimiento respecto al mes anterior

Calcula los ingresos aprobados por mes y el crecimiento porcentual respecto al mes anterior.

In [0]:
%sql
WITH ingresos_mes AS (
    SELECT
        DATE_TRUNC("MONTH", fecha_pago) AS mes,
        SUM(monto) AS ingresos_aprobados
    FROM pagos
    WHERE estado_pago = "aprobado"
    GROUP BY DATE_TRUNC("MONTH", fecha_pago)
),
comparacion AS (
    SELECT
        mes,
        ingresos_aprobados,
        LAG(ingresos_aprobados) OVER(ORDER BY mes) ingresos_anterior_mes
    FROM ingresos_mes
)
SELECT
    mes,
    ingresos_aprobados,
    ingresos_anterior_mes,
    100 * (ingresos_aprobados - ingresos_anterior_mes) / ingresos_anterior_mes AS crecimiento_pct
FROM comparacion
ORDER BY mes

### 23. Ranking de cursos por inscripciones dentro de cada categoría

Genera un ranking de cursos por cantidad de inscripciones dentro de cada categoría.

In [0]:
%sql
WITH inscripciones_por_curso AS (
    SELECT
        c.titulo_curso,
        c.categoria,
        COUNT(i.inscripcion_id) AS cantidad_inscripciones
    FROM cursos AS c
    LEFT JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
    GROUP BY c.titulo_curso, c.categoria
)
SELECT 
    titulo_curso,
    categoria,
    cantidad_inscripciones,
    DENSE_RANK() OVER(PARTITION BY categoria ORDER BY cantidad_inscripciones DESC) AS rank_categoria
FROM inscripciones_por_curso

### 24. Primera inscripción de cada estudiante

Obtén la primera inscripción de cada estudiante que tenga al menos una inscripción.

In [0]:
%sql
WITH inscripciones_rankeadas AS (
    SELECT
        inscripcion_id,
        estudiante_id,
        curso_id,
        fecha_inscripcion,
        ROW_NUMBER() OVER(PARTITION BY estudiante_id ORDER BY fecha_inscripcion) AS ranking
    FROM inscripciones
)
SELECT 
    ir.*,
    e.nombre_estudiante
FROM inscripciones_rankeadas AS ir 
LEFT JOIN estudiantes AS e
ON ir.estudiante_id = e.estudiante_id
WHERE ranking = 1


### 25. Estudiantes inscritos en al menos 3 categorías

Lista los estudiantes inscritos en al menos 3 categorías distintas.

In [0]:
%sql
SELECT
    i.estudiante_id,
    e.nombre_estudiante,
    COUNT(DISTINCT c.categoria) AS categorias_distintas
FROM estudiantes AS e
LEFT JOIN inscripciones AS i
ON e.estudiante_id = i.estudiante_id
LEFT JOIN cursos AS c
ON i.curso_id = c.curso_id
GROUP BY i.estudiante_id, e.nombre_estudiante
HAVING categorias_distintas >= 3
ORDER BY nombre_estudiante

### 26. Cursos con tasa de finalización superior a la global

Identifica los cursos cuya tasa de finalización es superior a la tasa de finalización global.

In [0]:
%sql
WITH tasa_global AS (
    SELECT
        ROUND(SUM(
            CASE
                WHEN estado = "completado" THEN 1
                ELSE 0
            END
        ) / COUNT(1), 2) AS tasa
    FROM inscripciones
),
tasa_curso AS (
    SELECT
        c.curso_id,
        c.titulo_curso,
        COUNT(i.inscripcion_id) AS total_inscripciones,
        ROUND(SUM(
            CASE
                WHEN estado = "completado" THEN 1
                ELSE 0
            END
        ), 2) AS total_completados,
        ROUND(SUM(
            CASE
                WHEN estado = "completado" THEN 1
                ELSE 0
            END
        ) / COUNT(1), 2) AS tasa_por_curso
    FROM cursos AS c
    LEFT JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
    GROUP BY c.curso_id, c.titulo_curso
)
SELECT *
FROM tasa_curso AS tc
CROSS JOIN tasa_global AS tg
WHERE tc.tasa_por_curso > (SELECT tasa FROM tasa_global)

### 27. Participación de cada instructor en los ingresos de su especialidad

Calcula el porcentaje de participación de cada instructor sobre los ingresos aprobados de su especialidad.

In [0]:
WITH ingresos_instructores AS (
    SELECT 
    ins.especialidad,
    ins.instructor_id,
    ins.nombre_instructor,
    SUM(p.monto) AS ingresos_aprobados
    FROM instructores AS ins
    LEFT JOIN cursos AS c
    ON ins.instructor_id = c.instructor_id
    LEFT JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
    LEFT JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id
    WHERE p.estado_pago = "aprobado"
    GROUP BY ins.especialidad, ins.instructor_id, ins.nombre_instructor
    ORDER BY nombre_instructor, instructor_id
)
SELECT
    especialidad,
    instructor_id,
    nombre_instructor,
    ingresos_aprobados,
    ROUND(100 * ingresos_aprobados / SUM(ingresos_aprobados) OVER(PARTITION BY especialidad), 2) AS contribucion
FROM ingresos_instructores
ORDER BY especialidad, contribucion DESC

### 28. Lifetime value y cuartil por estudiante

Calcula el valor de vida del estudiante (`lifetime_value`) usando pagos aprobados y clasifica a los estudiantes en 4 cuartiles con `NTILE(4)`.

In [0]:
%sql
WITH valor_estudiantes AS (
    SELECT
        e.estudiante_id,
        e.nombre_estudiante,
        SUM(p.monto) AS lifetime_value
    FROM estudiantes AS e
    LEFT JOIN inscripciones AS i
    ON e.estudiante_id = i.estudiante_id
    LEFT JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id
    WHERE p.estado_pago = "aprobado"
    GROUP BY e.estudiante_id, e.nombre_estudiante
)
SELECT
    estudiante_id,
    nombre_estudiante,
    lifetime_value,
    NTILE(4) OVER(ORDER BY lifetime_value DESC, estudiante_id) AS cuartil_valor
FROM valor_estudiantes

### 29. Porcentaje promedio de consumo por curso, limitado a 100% por inscripción

Calcula por curso el porcentaje promedio de consumo del contenido: minutos vistos divididos entre la duración del curso en minutos. Limita el resultado máximo de cada inscripción a 100%.

In [0]:
%sql
WITH minutos_inscripcion AS (
    SELECT 
        i.inscripcion_id,
        i.curso_id,
        SUM(s.minutos_vistos) AS minutos_vistos
    FROM inscripciones AS i
    LEFT JOIN sesiones AS s
    ON i.inscripcion_id = s.inscripcion_id
    GROUP BY i.inscripcion_id, i.curso_id
),
consumo AS (
    SELECT
        mi.inscripcion_id,
        mi.curso_id,
        LEAST(100 * mi.minutos_vistos / (c.duracion_horas * 60), 100) AS porcentaje_consumo
    FROM minutos_inscripcion AS mi
    LEFT JOIN cursos AS c
    ON c.curso_id = mi.curso_id
)
SELECT 
    c.curso_id,
    c.titulo_curso,
    ROUND(AVG(co.porcentaje_consumo), 2) AS promedio_pct_consumo
FROM cursos AS c
LEFT JOIN consumo AS co
ON c.curso_id = co.curso_id
GROUP BY c.curso_id, c.titulo_curso
ORDER BY promedio_pct_consumo DESC

### 30. Mayor pausa entre sesiones consecutivas por estudiante

Para cada estudiante, calcula la mayor cantidad de días de inactividad entre dos sesiones consecutivas. Muestra solo estudiantes cuya pausa máxima sea mayor a 14 días.

In [0]:
%sql
WITH sesion_estudiantes AS (
    SELECT
        e.estudiante_id,
        e.nombre_estudiante,
        s.fecha_sesion,
        LAG(s.fecha_sesion) OVER (PARTITION BY e.estudiante_id ORDER BY s.fecha_sesion, s.sesion_id) AS fecha_anterior
    FROM estudiantes AS e
    LEFT JOIN inscripciones AS i
    ON e.estudiante_id = i.estudiante_id
    LEFT JOIN sesiones AS s
    ON i.inscripcion_id = s.inscripcion_id
    ORDER BY estudiante_id, nombre_estudiante, fecha_sesion ASC
),
pausas AS (
    SELECT
     estudiante_id,
     nombre_estudiante,
     fecha_sesion,
     fecha_anterior,
     DATEDIFF(fecha_sesion, fecha_anterior) AS dias_inactividad
    FROM sesion_estudiantes
    WHERE fecha_anterior IS NOT NULL  
)

SELECT 
    estudiante_id,
    nombre_estudiante,
    MAX(dias_inactividad) AS pausa_maxima
FROM pausas
GROUP BY estudiante_id, nombre_estudiante
HAVING MAX(dias_inactividad) > 14